# BERT fine-tuning on LIAR (Shipli)
Runtime -> Change runtime type -> **T4 GPU** (Kaggle: Accelerator = GPU T4 x2, Internet = On).
Run the cells top to bottom. Upload `bert_bundle.zip` when asked.

In [ ]:
import torch; print(torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NO GPU')

In [ ]:
!pip install -q -U transformers datasets scikit-learn accelerate

In [ ]:
# Upload notebooks/bert_bundle.zip (Colab: left Files panel or this picker; Kaggle: add as dataset or upload to /kaggle/working)
import os, zipfile
try:
    from google.colab import files
    up = files.upload(); name = list(up)[0]
except ImportError:
    name = '/kaggle/working/bert_bundle.zip' if os.path.exists('/kaggle/working/bert_bundle.zip') else 'bert_bundle.zip'
zipfile.ZipFile(name).extractall('.')
print(os.listdir('.'))

## The code
The three source files for this experiment are shown below. Each cell writes the file to disk (`%%writefile`), so running it also reproduces the files the training cells import. Read them top to bottom: model, data fallback, training loop.

### `finetune/bert_model.py`
Model: bert-base-uncased with a classification head (text-only, or text + metadata)

In [ ]:
%%writefile finetune/bert_model.py
"""
finetune/bert_model.py
======================
Model + data helpers for fine-tuning a Hugging Face encoder (bert-base-uncased by
default; roberta-base works by changing --model-name) on LIAR.

Two input modes:
    text : the statement only
    meta : "speaker | party | subject | credit history" prefix + statement
           (built with the shared common.data_loader.build_metadata_text)
"""

from typing import Dict, List

import torch
from torch.utils.data import Dataset as TorchDataset
from transformers import AutoModelForSequenceClassification, AutoTokenizer

from common.config import NUM_CLASSES, IDX_TO_LABEL, LABEL_TO_IDX
from common.data_loader import build_metadata_text

DEFAULT_MODEL_NAME = "bert-base-uncased"


def build_texts(split, mode: str, sep_token: str = "[SEP]") -> List[str]:
    """
    Turn a LIAR split into the list of input strings for the chosen mode.

    The shared metadata builder joins prefix and statement with a literal "[SEP]".
    We swap that for the tokenizer's own separator so RoBERTa ("</s>") also works.
    """
    if mode == "text":
        return [(s or "").strip() for s in split["statement"]]
    if mode == "meta":
        return [build_metadata_text(row).replace(" [SEP] ", f" {sep_token} ") for row in split]
    raise ValueError(f"mode must be 'text' or 'meta', got {mode!r}")


class EncodedDataset(TorchDataset):
    """Pre-tokenised texts + project-order labels, ready for a DataLoader."""

    def __init__(self, texts: List[str], labels: List[int], tokenizer, max_len: int):
        self.enc = tokenizer(texts, truncation=True, max_length=max_len)
        self.labels = labels

    def __len__(self) -> int:
        return len(self.labels)

    def __getitem__(self, i: int) -> Dict:
        item = {k: v[i] for k, v in self.enc.items()}
        item["labels"] = self.labels[i]
        return item


def make_collate(tokenizer):
    """Pad each batch to its longest sequence (faster than padding everything to max_len)."""

    def collate(batch: List[Dict]) -> Dict[str, torch.Tensor]:
        labels = torch.tensor([b.pop("labels") for b in batch], dtype=torch.long)
        out = tokenizer.pad(batch, return_tensors="pt")
        out["labels"] = labels
        return out

    return collate


def load_model_and_tokenizer(model_name: str = DEFAULT_MODEL_NAME):
    """6-class classifier head on top of the pretrained encoder, in OUR label order."""
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    model = AutoModelForSequenceClassification.from_pretrained(
        model_name,
        num_labels=NUM_CLASSES,
        id2label=dict(IDX_TO_LABEL),
        label2id=dict(LABEL_TO_IDX),
    )
    return model, tokenizer


def count_params(model: torch.nn.Module) -> int:
    return sum(p.numel() for p in model.parameters())


### `finetune/liar_fallback.py`
LIAR data fallback loader

In [ ]:
%%writefile finetune/liar_fallback.py
"""
finetune/liar_fallback.py
=========================
Loads LIAR straight from the official release zip when `load_dataset("liar")` fails
(it is a script-based dataset, which recent `datasets` versions no longer support).

Produces the SAME splits (train 10,269 / val 1,284 / test 1,283), the same column
names and the same integer label ids as the Hugging Face copy, so the rest of the
pipeline (to_project_labels, build_metadata_text, shared test indices) is unchanged.
"""

import io
import os
import urllib.request
import zipfile
from typing import Dict, List

from common.data_loader import HF_LIAR_LABEL_NAMES

LIAR_ZIP_URL = "https://www.cs.ucsb.edu/~william/data/liar_dataset.zip"

COLUMNS = [
    "id", "label", "statement", "subject", "speaker", "job_title", "state_info",
    "party_affiliation", "barely_true_counts", "false_counts", "half_true_counts",
    "mostly_true_counts", "pants_on_fire_counts", "context",
]
COUNT_COLUMNS = [c for c in COLUMNS if c.endswith("_counts")]


def parse_tsv(text: str) -> Dict[str, List]:
    """Parse one LIAR .tsv file into {column: list}. Labels become HF integer ids."""
    cols: Dict[str, List] = {c: [] for c in COLUMNS}
    for line in text.splitlines():
        if not line.strip():
            continue
        parts = line.split("\t")
        parts = (parts + [""] * len(COLUMNS))[: len(COLUMNS)]   # tolerate short rows
        for name, value in zip(COLUMNS, parts):
            if name == "label":
                value = HF_LIAR_LABEL_NAMES.index(value.strip().lower())
            elif name in COUNT_COLUMNS:
                value = float(value) if value.strip() else 0.0
            elif name == "id":
                value = value.replace(".json", "")
            cols[name].append(value)
    return cols


TSV_FILES = ("train.tsv", "valid.tsv", "test.tsv")


def _read_tsv_texts(cache_dir: str) -> list:
    """Return the text of train/valid/test .tsv.

    Prefers the original files committed in the repo (LIAR_TSV_DIR, default `liar_data/`);
    only downloads the official zip if they are missing.
    """
    tsv_dir = os.environ.get("LIAR_TSV_DIR", "liar_data")
    if all(os.path.exists(os.path.join(tsv_dir, f)) for f in TSV_FILES):
        print(f"Reading LIAR from {tsv_dir}/")
        return [open(os.path.join(tsv_dir, f), encoding="utf-8").read() for f in TSV_FILES]

    os.makedirs(cache_dir, exist_ok=True)
    zpath = os.path.join(cache_dir, "liar_dataset.zip")
    if not os.path.exists(zpath):
        print(f"Downloading LIAR from {LIAR_ZIP_URL} ...")
        urllib.request.urlretrieve(LIAR_ZIP_URL, zpath)
    with zipfile.ZipFile(zpath) as z:
        return [io.TextIOWrapper(io.BytesIO(z.read(f)), encoding="utf-8").read() for f in TSV_FILES]


def load_liar_from_zip(cache_dir: str = "data/"):
    """Return (train, val, test) as `datasets.Dataset` objects."""
    from datasets import ClassLabel, Dataset, Features, Value

    feats = Features({
        c: (ClassLabel(names=HF_LIAR_LABEL_NAMES) if c == "label"
            else Value("float32") if c in COUNT_COLUMNS else Value("string"))
        for c in COLUMNS
    })
    return tuple(Dataset.from_dict(parse_tsv(t), features=feats) for t in _read_tsv_texts(cache_dir))


### `finetune/train_bert.py`
Training and evaluation script (`--mode text|meta`)

In [ ]:
%%writefile finetune/train_bert.py
"""
finetune/train_bert.py  --  BERT fine-tuning on LIAR (Run 1: text, Run 2: meta)
================================================================================

HOW TO RUN (from the repository root, ideally on a Colab/Kaggle T4):
    python -m finetune.train_bert --mode text     # Run 1: statement only
    python -m finetune.train_bert --mode meta     # Run 2: metadata + statement
    python -m finetune.train_bert --mode text --epochs 1 --max-train 200   # smoke test

    # RoBERTa (Sandya): same script, different model + result name
    python -m finetune.train_bert --mode text --model-name roberta-base --run-name roberta_text

OUTPUT FILES (same schema as the LSTM, plus GPU-memory / per-1k-time extras):
    results/<run-name>.json         default run names: bert_text, bert_meta
    results/<run-name>_preds.csv    id,y_true,y_pred,latency_ms on the shared 300-row subset
    checkpoints/<run-name>.pt       best weights (git-ignored)
"""

import argparse
import os
import time
from typing import List, Tuple

import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from transformers import get_linear_schedule_with_warmup

from baselines.train import set_seed, to_project_labels, shared_test_indices
from common.config import (
    BATCH_SIZE_EVAL, BATCH_SIZE_TRAIN, CHECKPOINTS_DIR, LABEL_TO_IDX,
    LEARNING_RATE_DEFAULT, MAX_LENGTH_TEXT, NUM_EPOCHS_DEFAULT, RANDOM_SEED, RESULTS_DIR,
)
from common.data_loader import load_liar
from common.evaluate import build_results_dict, evaluate, save_predictions_csv, save_results_json
from finetune.bert_model import (
    DEFAULT_MODEL_NAME, EncodedDataset, build_texts, count_params,
    load_model_and_tokenizer, make_collate,
)

T4_PRICE_PER_HOUR = 0.35   # same cost assumption as every other model
OVERHEAD = 1.10


def _sync(device: torch.device) -> None:
    """Wait for queued GPU work so timings are honest."""
    if device.type == "cuda":
        torch.cuda.synchronize()


@torch.no_grad()
def predict(model, loader, device, use_amp: bool) -> List[int]:
    model.eval()
    preds: List[int] = []
    for batch in loader:
        batch = {k: v.to(device) for k, v in batch.items() if k != "labels"}
        with torch.autocast(device_type=device.type, dtype=torch.float16, enabled=use_amp):
            logits = model(**batch).logits
        preds.extend(logits.argmax(dim=-1).tolist())
    return preds


@torch.no_grad()
def predict_with_latency(model, texts, tokenizer, max_len, device, use_amp) -> Tuple[List[int], List[float]]:
    """One statement at a time (batch size 1), like the demo app, timed per statement."""
    model.eval()
    preds, lat = [], []
    # Warm-up so the first timed call doesn't include CUDA start-up cost.
    warm = tokenizer(texts[0], truncation=True, max_length=max_len, return_tensors="pt").to(device)
    model(**warm)
    for t in texts:
        _sync(device)
        t0 = time.perf_counter()
        enc = tokenizer(t, truncation=True, max_length=max_len, return_tensors="pt").to(device)
        with torch.autocast(device_type=device.type, dtype=torch.float16, enabled=use_amp):
            logits = model(**enc).logits
        preds.append(int(logits.argmax(dim=-1)))
        _sync(device)
        lat.append((time.perf_counter() - t0) * 1000)
    return preds, lat


def main():
    ap = argparse.ArgumentParser(description="Fine-tune an encoder on LIAR")
    ap.add_argument("--mode", choices=["text", "meta"], required=True)
    ap.add_argument("--model-name", default=DEFAULT_MODEL_NAME)
    ap.add_argument("--run-name", default=None, help="results file stem (default bert_<mode>)")
    ap.add_argument("--epochs", type=int, default=NUM_EPOCHS_DEFAULT)
    ap.add_argument("--patience", type=int, default=2, help="early stopping on val macro-F1")
    ap.add_argument("--batch-size", type=int, default=BATCH_SIZE_TRAIN)
    ap.add_argument("--lr", type=float, default=LEARNING_RATE_DEFAULT)
    ap.add_argument("--max-len", type=int, default=None,
                    help=f"default {MAX_LENGTH_TEXT} for text, 2x for meta")
    ap.add_argument("--warmup", type=float, default=0.1, help="warm-up fraction of total steps")
    ap.add_argument("--weight-decay", type=float, default=0.01)
    ap.add_argument("--no-class-weights", action="store_true")
    ap.add_argument("--no-fp16", action="store_true")
    ap.add_argument("--max-train", type=int, default=None, help="use only N train rows (smoke test)")
    args = ap.parse_args()

    run_name = args.run_name or f"bert_{args.mode}"
    max_len = args.max_len or (MAX_LENGTH_TEXT if args.mode == "text" else MAX_LENGTH_TEXT * 2)

    set_seed(RANDOM_SEED)
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    use_amp = device.type == "cuda" and not args.no_fp16
    gpu_name = torch.cuda.get_device_name(0) if device.type == "cuda" else "CPU"
    print(f"Device: {gpu_name} | fp16: {use_amp} | mode: {args.mode} | model: {args.model_name}")

    # ---------- data (same split, same label order as every other model) ----------
    try:
        train, val, test = load_liar()
    except Exception as e:   # script-based "liar" is unsupported on recent `datasets`
        print(f"load_liar() failed ({e}); falling back to the official LIAR zip")
        from finetune.liar_fallback import load_liar_from_zip
        train, val, test = load_liar_from_zip()
    print(f"Train {len(train)} | val {len(val)} | test {len(test)}")
    y_train, y_val, y_test = to_project_labels(train), to_project_labels(val), to_project_labels(test)
    if args.max_train:
        train = train.select(range(args.max_train))
        y_train = y_train[: args.max_train]

    model, tokenizer = load_model_and_tokenizer(args.model_name)
    sep = tokenizer.sep_token or "[SEP]"
    texts = {n: build_texts(s, args.mode, sep) for n, s in (("train", train), ("val", val), ("test", test))}
    print("Example input:", texts["train"][0][:200])

    collate = make_collate(tokenizer)
    mk = lambda name, ys, bs, shuffle: DataLoader(
        EncodedDataset(texts[name], ys, tokenizer, max_len),
        batch_size=bs, shuffle=shuffle, collate_fn=collate)
    g = torch.Generator().manual_seed(RANDOM_SEED)
    train_loader = DataLoader(EncodedDataset(texts["train"], y_train, tokenizer, max_len),
                              batch_size=args.batch_size, shuffle=True, collate_fn=collate, generator=g)
    val_loader = mk("val", y_val, BATCH_SIZE_EVAL, False)
    test_loader = mk("test", y_test, BATCH_SIZE_EVAL, False)

    model.to(device)
    n_params = count_params(model)
    print(f"Parameters: {n_params:,}")

    # ---------- loss / optimiser / schedule ----------
    weights = None
    if not args.no_class_weights:
        counts = np.bincount(y_train, minlength=len(LABEL_TO_IDX))
        weights = torch.tensor(counts.sum() / (len(counts) * counts), dtype=torch.float, device=device)
    loss_fn = nn.CrossEntropyLoss(weight=weights)
    no_decay = ("bias", "LayerNorm.weight")
    groups = [
        {"params": [p for n, p in model.named_parameters() if not any(k in n for k in no_decay)],
         "weight_decay": args.weight_decay},
        {"params": [p for n, p in model.named_parameters() if any(k in n for k in no_decay)],
         "weight_decay": 0.0},
    ]
    opt = torch.optim.AdamW(groups, lr=args.lr)
    total_steps = len(train_loader) * args.epochs
    sched = get_linear_schedule_with_warmup(opt, int(args.warmup * total_steps), total_steps)
    scaler = torch.cuda.amp.GradScaler(enabled=use_amp)

    # ---------- training with early stopping on val macro-F1 ----------
    os.makedirs(CHECKPOINTS_DIR, exist_ok=True)
    ckpt_path = os.path.join(CHECKPOINTS_DIR, f"{run_name}.pt")
    if device.type == "cuda":
        torch.cuda.reset_peak_memory_stats()
    best_f1, best_epoch, bad = -1.0, 0, 0
    t_start = time.time()
    for epoch in range(1, args.epochs + 1):
        model.train()
        running = 0.0
        for batch in train_loader:
            batch = {k: v.to(device) for k, v in batch.items()}
            labels = batch.pop("labels")
            opt.zero_grad(set_to_none=True)
            with torch.autocast(device_type=device.type, dtype=torch.float16, enabled=use_amp):
                logits = model(**batch).logits
            loss = loss_fn(logits.float(), labels)
            scaler.scale(loss).backward()
            scaler.unscale_(opt)
            nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            scaler.step(opt)
            scaler.update()
            sched.step()
            running += loss.item() * len(labels)
        val_m = evaluate(y_val, predict(model, val_loader, device, use_amp))
        print(f"epoch {epoch} | loss {running / len(train_loader.dataset):.4f} | "
              f"val acc {val_m['accuracy']:.4f} macro-F1 {val_m['macro_f1']:.4f}")
        if val_m["macro_f1"] > best_f1:
            best_f1, best_epoch, bad = val_m["macro_f1"], epoch, 0
            torch.save({"state_dict": model.state_dict(), "args": vars(args)}, ckpt_path)
        else:
            bad += 1
            if bad >= args.patience:
                print(f"Early stopping (best epoch {best_epoch})")
                break
    _sync(device)
    train_minutes = (time.time() - t_start) / 60
    train_mem_mb = torch.cuda.max_memory_allocated() / 2**20 if device.type == "cuda" else 0.0

    # ---------- final evaluation with the best epoch ----------
    model.load_state_dict(torch.load(ckpt_path, map_location=device)["state_dict"])
    full_m = evaluate(y_test, predict(model, test_loader, device, use_amp))

    idx = shared_test_indices(test)
    sub_texts, sub_y = [texts["test"][i] for i in idx], [y_test[i] for i in idx]
    if device.type == "cuda":
        torch.cuda.reset_peak_memory_stats()
    sub_pred, lat = predict_with_latency(model, sub_texts, tokenizer, max_len, device, use_amp)
    infer_mem_mb = torch.cuda.max_memory_allocated() / 2**20 if device.type == "cuda" else 0.0
    sub_m = evaluate(sub_y, sub_pred)

    avg_lat = float(np.mean(lat))
    cost_per_1k = avg_lat / 1000 / 3600 * 1000 * T4_PRICE_PER_HOUR * OVERHEAD

    results = build_results_dict(
        model_name=run_name,
        metrics=sub_m,
        params=n_params,
        train_minutes=round(train_minutes, 2),
        gpu=gpu_name,
        avg_latency_ms=round(avg_lat, 3),
        est_cost_per_1k=round(cost_per_1k, 6),
        cost_assumption="Priced at T4 rate $0.35/hour incl. 10% overhead; batch size 1 inference",
        notes=f"{args.model_name}, mode={args.mode}, best epoch {best_epoch}, "
              f"class-weighted loss={not args.no_class_weights}, fp16={use_amp}, max_len={max_len}",
    )
    # Extra fields requested in the project brief (additive, schema stays compatible):
    results["inference_sec_per_1000"] = round(avg_lat, 3)   # ms/statement == seconds per 1,000
    results["infer_seconds_per_1000"] = round(avg_lat, 3)   # same key the LLM results / cost_analysis use
    results["gpu_peak_mem_mb"] = round(max(train_mem_mb, infer_mem_mb), 1)
    results["gpu_mem_train_mb"] = round(train_mem_mb, 1)
    results["gpu_mem_inference_mb"] = round(infer_mem_mb, 1)
    results["full_test"] = {"accuracy": full_m["accuracy"], "macro_f1": full_m["macro_f1"],
                            "n": len(y_test), "confusion_matrix": full_m["confusion_matrix"]}
    results["val_best_macro_f1"] = best_f1
    results["hyperparams"] = vars(args)

    save_results_json(results, run_name, RESULTS_DIR)
    save_predictions_csv(idx, sub_y, sub_pred, lat, run_name, RESULTS_DIR)
    print(f"Subset ({len(idx)}): acc {sub_m['accuracy']:.4f} macro-F1 {sub_m['macro_f1']:.4f} | "
          f"full test: acc {full_m['accuracy']:.4f} macro-F1 {full_m['macro_f1']:.4f} | "
          f"latency {avg_lat:.1f} ms | train {train_minutes:.1f} min | "
          f"GPU mem train {train_mem_mb:.0f} MB / infer {infer_mem_mb:.0f} MB")


if __name__ == "__main__":
    main()


In [ ]:
!python -m finetune.train_bert --mode text --epochs 1 --max-train 200   # quick smoke test (~1 min). Check it prints metrics, no errors.

In [ ]:
!python -m finetune.train_bert --mode text    # Run 1: statement only

In [ ]:
!python -m finetune.train_bert --mode meta    # Run 2: metadata + statement

In [ ]:
import json
for r in ('bert_text','bert_meta'):
    d = json.load(open(f'results/{r}.json'))
    print(r, {k: d[k] for k in ('accuracy','macro_f1','params','train_minutes','inference_sec_per_1000','gpu_mem_train_mb','gpu_mem_inference_mb')})
    print('  full test:', d['full_test']['accuracy'], d['full_test']['macro_f1'])

In [ ]:
# Download the result files (then put them in the shared Drive folder)
!zip -r bert_results.zip results/bert_text.json results/bert_text_preds.csv results/bert_meta.json results/bert_meta_preds.csv
try:
    from google.colab import files; files.download('bert_results.zip')
except ImportError:
    print('Kaggle: download bert_results.zip from the Output panel')